In [63]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Extraer información de la base de datos

In [64]:
# Cargar primera base de datos
raw_data = pd.read_excel('datos/DATOS HISTÓRICOS 2023_2024_TODAS ESTACIONES_ITESM.xlsx', header = None)
raw_data.shape 

# Tomar las primeras 3 filas como cabecera
header = raw_data.iloc[0:3].astype(str)   # convertir a string por si hay números
# Concatenar por columna, ignorando NaN
new_columns = header.apply(lambda x: " ".join(x.dropna()), axis=0)

# Redefinir columnas
raw_data.columns = new_columns

# Quitar las filas de metadatos
raw_data = raw_data.iloc[3:].reset_index(drop=True)

In [65]:
raw_data.columns

Index(['nan nan date', 'SURESTE CO ppm', 'SURESTE NO ppb', 'SURESTE NO2 ppb',
       'SURESTE NOX ppb', 'SURESTE O3 ppb', 'SURESTE PM10 ug/m3',
       'SURESTE PM2.5 ug/m3', 'SURESTE PRS mmhg', 'SURESTE RAINF mm/hr',
       ...
       'NOROESTE 3 PM10 ug/m3', 'NOROESTE 3 PM2.5 ug/m3',
       'NOROESTE 3 PRS mmhg', 'NOROESTE 3 RAINF mm/hr', 'NOROESTE 3 RH %',
       'NOROESTE 3 SO2 ppb', 'NOROESTE 3 SR KW/m2', 'NOROESTE 3 TOUT degC',
       'NOROESTE 3 WSR KMPH', 'NOROESTE 3 WDR DEG'],
      dtype='object', length=240)

In [66]:
# Cargar segunda base de datos
ex_rest_24 = pd.read_excel('datos/DATOS HISTÓRICOS 2024_TODAS ESTACIONES.xlsx', sheet_name=None)
rest_24 = pd.concat(ex_rest_24.values(), axis=1)
rest_24.shape

(8784, 240)

# Limpieza, Tipo de Datos, Valores Faltantes

In [ ]:
# Eliminar columnas vacias o redundantes
raw_data.dropna(how='all', axis=1, inplace=True)
rest_24.drop(columns='Fecha y hora', inplace=True)

# Establecer nans y cambiar nombre de la primera columna
raw_data.replace("", np.nan, inplace=True)
raw_data.rename(columns={'nan nan date': 'Date'}, inplace=True)

# Resetear index
raw_data = raw_data.reset_index(drop=True)

# Hacer datetime columna Date
rest_24['Date'] = pd.to_datetime(rest_24['Date'])
raw_data["Date"] = pd.to_datetime(raw_data["Date"])


# Hacer Columnas Numéricas
for col in raw_data.columns:
    if col != "Date":
        raw_data[col] = pd.to_numeric(raw_data[col], errors="coerce")

# Resetear el indice como la fecha
raw_data.set_index("Date", inplace=True)
rest_24.set_index('Date', inplace=True)

print(raw_data.shape)
print(rest_24.shape)

# # Combinar ambos dataframes en uno
# nombre_columnas = raw_data.columns
# rest_24_1 = rest_24.loc['2024-08-01 01:00:00':]
# rest_24_1.columns = nombre_columnas
# raw_data = pd.concat([raw_data, rest_24_1], axis=0)

# # Revisar valores nulos
# print(raw_data.shape)
# raw_data.isna().sum()


(13870, 224)
(8784, 225)


In [69]:
raw_data.to_csv('datos/raw1.csv', index=True)
rest_24.to_csv('datos/rest_24.csv', index=True)


In [68]:
# Combinar ambos dataframes en uno
nombre_columnas = raw_data.columns
rest_24_1 = rest_24.loc['2024-08-01 01:00:00':]
rest_24_1.columns = nombre_columnas
raw_data = pd.concat([raw_data, rest_24_1], axis=0)

# Revisar valores nulos
print(raw_data.shape)
raw_data.isna().sum()

ValueError: Length mismatch: Expected axis has 225 elements, new values have 224 elements

In [ ]:
# Mostrar el dataset
raw_data.head()

In [ ]:
# Mostrar el dataset
raw_data.tail()

In [ ]:
# Primero utilizamos interpolación de datos faltantes en espacios más pequeños, usando una interpolación lineal para no más de 3 datos faltantes consecutivos
data_interpolate = (
    raw_data
    .interpolate(method="time", limit=3, limit_direction="both", limit_area="inside")
)

data_interpolate.isna().sum()

In [ ]:
# Para rellenar espacios grandes de missing values utilizando un KNNIMPUTE 
from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors=5)
df_imputed = pd.DataFrame(imputer.fit_transform(data_interpolate), columns=data_interpolate.columns, index=data_interpolate.index)
df_imputed.isna().sum()

In [ ]:
# Ejemplo de los datos en una variable 
plt.figure(figsize=(15,6))
plt.plot(df_imputed['SURESTE.6'])
plt.title('SURESTE.6')
plt.ylabel('Concentración')
plt.xlabel('Date')
plt.show()

In [ ]:
df_imputed.to_csv('datos/datos_limpios.csv', index=True)